# Интерпретируемая классификация токсичности на нескольких языках

**Задача:** оценить предобученный мультиязычный классификатор и показать, какие слова сильнее всего влияют на его решение.

**Стек:** Transformers, PyTorch, scikit-learn, pandas, seaborn.

Для интерпретации используется occlusion: слова по одному исключаются из текста, после чего измеряется изменение вероятности токсичного класса.


Модель `unitary/multilingual-toxic-xlm-roberta` обучалась на английском, французском, испанском, итальянском, португальском, турецком и русском языках. В MultiHateCheck присутствуют пять из них, поэтому агрегированные метрики считаются только для `eng`, `fra`, `spa`, `ita`, `por`. Это исключает некорректное сравнение на языках вне заявленной области применения модели.


In [ ]:
import html
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from datasets import load_dataset
from IPython.display import HTML, display
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
    roc_auc_score,
)
from transformers import AutoModelForSequenceClassification, AutoTokenizer

MODEL_NAME = "unitary/multilingual-toxic-xlm-roberta"
SUPPORTED_LANGUAGES = ["eng", "fra", "spa", "ita", "por"]
MAX_PER_LANGUAGE = 300
BATCH_SIZE = 32
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
dataset = load_dataset("mteb/multi-hatecheck", split="test")
data = dataset.to_pandas()
data["label"] = (data["is_hateful"].astype(str).str.lower() == "hateful").astype(int)
data = data[data["lang"].isin(SUPPORTED_LANGUAGES)].copy()

evaluation_data = (
    data.groupby("lang", group_keys=False)
    .apply(lambda part: part.sample(min(MAX_PER_LANGUAGE, len(part)), random_state=SEED))
    .reset_index(drop=True)
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()

print(evaluation_data.groupby(["lang", "label"]).size().unstack(fill_value=0))
print(f"Устройство: {DEVICE}; выходов модели: {model.config.num_labels}")


In [ ]:
def predict_toxicity(texts: list[str]) -> np.ndarray:
    probabilities = []
    for start in range(0, len(texts), BATCH_SIZE):
        batch = texts[start : start + BATCH_SIZE]
        encoded = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=256,
        ).to(DEVICE)
        with torch.inference_mode():
            logits = model(**encoded).logits

        if logits.shape[1] == 1:
            batch_probabilities = torch.sigmoid(logits[:, 0])
        else:
            toxic_index = next(
                (
                    index
                    for index, label in model.config.id2label.items()
                    if "toxic" in str(label).lower()
                ),
                1,
            )
            batch_probabilities = torch.softmax(logits, dim=-1)[:, toxic_index]
        probabilities.extend(batch_probabilities.cpu().numpy())
    return np.asarray(probabilities)


evaluation_data["toxicity_probability"] = predict_toxicity(
    evaluation_data["text"].astype(str).tolist()
)
evaluation_data["prediction"] = (
    evaluation_data["toxicity_probability"] >= 0.5
).astype(int)


In [ ]:
print(
    classification_report(
        evaluation_data["label"],
        evaluation_data["prediction"],
        target_names=["non-hateful", "hateful"],
        zero_division=0,
    )
)
print(
    "ROC-AUC:",
    round(
        roc_auc_score(
            evaluation_data["label"],
            evaluation_data["toxicity_probability"],
        ),
        4,
    ),
)

cm = confusion_matrix(evaluation_data["label"], evaluation_data["prediction"])
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["non-hateful", "hateful"],
    yticklabels=["non-hateful", "hateful"],
)
plt.xlabel("Предсказание")
plt.ylabel("Истинная метка")
plt.title("Матрица ошибок")
plt.tight_layout()


In [ ]:
per_language = []
for language, part in evaluation_data.groupby("lang"):
    precision, recall, f1, _ = precision_recall_fscore_support(
        part["label"],
        part["prediction"],
        average="binary",
        zero_division=0,
    )
    per_language.append(
        {
            "language": language,
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "samples": len(part),
        }
    )

language_metrics = pd.DataFrame(per_language).sort_values("f1", ascending=False)
language_metrics.round(4)


In [ ]:
def occlusion_importance(text: str) -> tuple[float, list[tuple[str, float]]]:
    words = re.findall(r"\S+", str(text))
    if not words:
        return 0.0, []

    variants = [str(text)] + [
        " ".join(words[:index] + words[index + 1 :])
        for index in range(len(words))
    ]
    probabilities = predict_toxicity(variants)
    base_probability = float(probabilities[0])
    importance = np.maximum(base_probability - probabilities[1:], 0.0)
    scale = importance.max() or 1.0
    normalized = importance / scale
    return base_probability, list(zip(words, normalized.tolist()))


def highlight_words(weighted_words: list[tuple[str, float]]) -> HTML:
    fragments = []
    for word, weight in weighted_words:
        safe_word = html.escape(word)
        alpha = 0.10 + 0.75 * weight
        fragments.append(
            f'<span style="background: rgba(220, 38, 38, {alpha:.2f}); '
            f'padding: 2px 3px; border-radius: 3px">{safe_word}</span>'
        )
    return HTML(" ".join(fragments))


In [ ]:
examples = evaluation_data[evaluation_data["label"] == 1].groupby("lang").head(1)
for row in examples.itertuples(index=False):
    probability, weighted_words = occlusion_importance(row.text)
    print(f"{row.lang}: toxicity_probability={probability:.3f}")
    display(highlight_words(weighted_words))


## Ограничения

Occlusion показывает чувствительность модели, а не причинность. Метод дорогой по времени, потому что для текста из `n` слов выполняется `n + 1` инференсов. Для больших объёмов можно использовать Integrated Gradients или SHAP, сохранив проверку на небольшом наборе вручную размеченных примеров.
